In [1]:
import os
import re
import pandas as pd
import plotly.express as px
import sys


In [2]:
datapath = r"/Volumes/TytellLab$/NewData/ElongateFoils/ElongateFoil_DorsalandAnalFins"
pivdatapath = os.path.join(datapath, "Processed data")

In [3]:
pivfiles = []
for root, dirs, files in os.walk(pivdatapath):
    for file in files:
        if file.endswith(".txt"):
            pivfiles.append(os.path.join(root, file))

print(f"Found {len(pivfiles)} PIV files.")

Found 160 PIV files.


In [4]:
pivfiles[50]

'/Volumes/TytellLab$/NewData/ElongateFoils/ElongateFoil_DorsalandAnalFins/Processed data/2026-06-22/flow15_0/flow15_0_0001_0010.txt'

In [5]:
data = []
for fn1 in pivfiles:
    # We need to handle double numbers in the filename
    # flow3_0_0001.txt or flow3_0_0001_0002.txt
    # the last number is the frame number, and if there is another four digit number, it means
    # that we ran two PIV analyses on the same image, and the second one is the one we want to use.
    m = re.search(r"(\d{4})?_(\d{4})\.txt", fn1)
    if not m:
        print(f"Could not find frame number in filename: {fn1}")
        continue

    frame_number = int(m.groups()[1])
    if m.groups()[0] is None:
        rep = 1
    else:
        rep = 2

    data1 = pd.read_csv(fn1, header=2,
                        names=["x", "y", "u", "v", "vectype"])
    data1["filename"] = os.path.basename(fn1)
    data1["frame_number"] = frame_number
    data1["rep"] = rep
    
    m = re.search(r"flow(\d+_\d+)", fn1)
    if m:
        ms1 = re.sub("_", ".", m.groups()[0])
        data1["motor_speed"] = float(ms1)
    else:
        print(f"Could not find motor speed in filename: {fn1}")
        data1["motor_speed"] = None
    
    data.append(data1)

data = pd.concat(data, ignore_index=True)


Get only the last rep

In [6]:
data = data[
    data['rep'] == data.groupby(["motor_speed"])["rep"].transform("max")
    ]

In [7]:
data.head()

,x,y,u,v,vectype,filename,frame_number,rep,motor_speed
66,0.05606,0.029797,NaN,NaN,0,flow11_0_0001_0001.txt,1,2,11.0
67,0.05606,0.037878,NaN,NaN,0,flow11_0_0001_0001.txt,1,2,11.0
68,0.05606,0.045959,NaN,NaN,0,flow11_0_0001_0001.txt,1,2,11.0
69,0.05606,0.054040,NaN,NaN,0,flow11_0_0001_0001.txt,1,2,11.0
70,0.05606,0.062120,NaN,NaN,0,flow11_0_0001_0001.txt,1,2,11.0


Only keep vectype == 1, which means it's a good vector

In [8]:
data = data[data["vectype"] == 1]

Average over x values

In [9]:
meandata = data.groupby(["motor_speed", "y"])[["x", "u", "v"]].median(skipna=True, numeric_only=True)

In [10]:
meandata.head()

x         u         v
motor_speed y                                     
3.0         0.029797  0.076261 -0.032409  0.000012
            0.037878  0.076261 -0.043457 -0.000607
            0.045959  0.076261 -0.040809  0.001137
            0.054040  0.076261 -0.032955  0.000287
            0.062120  0.076261 -0.040104  0.000773

Get rid of the nested index for the rows

In [11]:
meandata.reset_index(inplace=True)

In [12]:
meandata.head()

,motor_speed,y,x,u,v
0,3.0,0.029797,0.076261,-0.032409,0.000012
1,3.0,0.037878,0.076261,-0.043457,-0.000607
2,3.0,0.045959,0.076261,-0.040809,0.001137
3,3.0,0.054040,0.076261,-0.032955,0.000287
4,3.0,0.062120,0.076261,-0.040104,0.000773


In [13]:
fig = px.line(meandata, x="y", y="u", color=meandata["motor_speed"])
             # color_continuous_scale='Viridis') # doesn't work even though Claude said it would
fig.show()


Here's the overall mean of medians for each flow speed

In [14]:
framemed = data.groupby(["motor_speed", "frame_number"])[["u"]].median()

means = framemed.groupby("motor_speed")["u"].mean(skipna=True)

# turn the means into a dataframe so they can be added to csv
means = means.reset_index()

print(means)

   motor_speed         u
0          3.0 -0.040178
1          5.0 -0.016090
2          7.0 -0.013009
3          9.0 -0.137035
4         11.0 -0.185876
5         13.0 -0.205644
6         15.0 -0.256220
7         17.0 -0.275099
8         19.0 -0.343897


Write in excel force data as CSV 

In [15]:
df = pd.read_csv(r"/Volumes/TytellLab$/NewData/ElongateFoils/ElongateFoil_DorsalandAnalFins/Raw data/Force data/2026-06-23/cylinderflowtest.csv")

# remove empty columns and rows
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df.dropna(axis = 1, how='all')
df = df.dropna(axis = 0, how='all')


# Add column for flowspeed corresponding with motor speed
for speed in df["Motor Speed"]:
    for motor in means["motor_speed"]:
        if speed == motor:
            df.loc[df["Motor Speed"] == speed, "Flow Speed"] = means.loc[means["motor_speed"] == motor, "u"].values[0]

print(df)

   Motor Speed   Y force  Y force SD  X torque  X torque SD Video name  \
0          3.0 -0.003267    0.034215 -0.001018     0.000936    flow3_0   
1          5.0 -0.006725    0.030996 -0.001835     0.000815    flow5_0   
2          7.0  0.007828    0.028939 -0.003579     0.000795    flow7_0   
3          9.0  0.025189    0.028019 -0.006198     0.000855    flow9_0   
4         11.0  0.048413    0.028595 -0.009596     0.001026   flow11_0   
5         13.0  0.074219    0.028900 -0.013490     0.001262   flow13_0   
6         15.0  0.103885    0.029787 -0.017940     0.001372   flow15_0   
7         17.0  0.145726    0.031111 -0.024278     0.001642   flow17_0   
8         19.0  0.216083    0.033380 -0.035126     0.002046   flow19_0   

   Force data file name  Flow Speed  
0  cylinder_flowtest014   -0.040178  
1  cylinder_flowtest004   -0.016090  
2  cylinder_flowtest006   -0.013009  
3  cylinder_flowtest007   -0.137035  
4  cylinder_flowtest008   -0.185876  
5  cylinder_flowtest010   -0.20

Create a visualization of flowspeed vs. y force

In [16]:
!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install statsmodels

Looking in links: /var/folders/24/xd9pgpbn4lxf3mxt7blq5y3w0000gn/T/tmpvvq9qv0u


In [17]:

fig = px.scatter(df, x=df["Y force"], y=df["Flow Speed"], labels={"Y force": "Y Force (N)", "Flow Speed": "Flow Speed (m/s)"}, title="Flow Speed vs Y Force", trendline="lowess", trendline_color_override="red")

fig.show()

In [18]:
# save new dataframe with flow speed added to csv
df.to_csv(r"/Volumes/TytellLab$/NewData/ElongateFoils/ElongateFoil_DorsalandAnalFins/Raw data/Force data/2026-06-23/cylinderflowtest_with_flowspeed.csv", index=False)